In [ ]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 MB 13.0 MB/s eta 0:00:0000:0100:01


In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/credit_card.csv")

In [ ]:
# keep only the types where fraud occurs, drop useless columns
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
df = df.drop(columns=["nameOrig", "nameDest", "isFlaggedFraud"])
print(df.shape, df["isFraud"].mean())

(2770409, 8) 0.002964544224336551


In [3]:
df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
df["origEmptied"] = ((df["newbalanceOrig"] == 0) & (df["oldbalanceOrg"] > 0)).astype(int)
df["destWasEmpty"] = (df["oldbalanceDest"] == 0).astype(int)
df["amountLog"] = np.log1p(df["amount"])
df["hourOfDay"] = df["step"] % 24

In [5]:
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,errorBalanceOrig,errorBalanceDest,origEmptied,destWasEmpty,amountLog,hourOfDay
2,1,TRANSFER,181.00,181.0,0.0,0.0,0.00,1,0.00,181.0,1,1,5.204007,1
3,1,CASH_OUT,181.00,181.0,0.0,21182.0,0.00,1,0.00,21363.0,1,0,5.204007,1
15,1,CASH_OUT,229133.94,15325.0,0.0,5083.0,51513.44,0,213808.94,182703.5,1,0,12.342066,1
19,1,TRANSFER,215310.30,705.0,0.0,22425.0,0.00,0,214605.30,237735.3,1,0,12.279840,1
24,1,TRANSFER,311685.89,10835.0,0.0,6267.0,2719172.89,0,300850.89,-2401220.0,1,0,12.649754,1


In [4]:
df.groupby("isFraud")[["errorBalanceOrig", "errorBalanceDest", "origEmptied", "destWasEmpty"]].mean()

,errorBalanceOrig,errorBalanceDest,origEmptied,destWasEmpty
isFraud,,,,
0,286803.509954,-30910.325352,0.427219,0.139009
1,10692.325265,732509.301069,0.975527,0.651528


In [ ]:
# One hot encode the type column
df["type_TRANSFER"] = (df["type"] == "TRANSFER").astype(int)
df = df.drop(columns=["type"])

In [ ]:
#time based split
split_step = df["step"].quantile(0.8) # 80% train , 20% test

train = df[df["step"] <= split_step]
test = df[df["step"] > split_step]

X_train = train.drop(columns=["isFraud", "step"])
y_train = train["isFraud"]
X_test = test.drop(columns=["isFraud", "step"])
y_test = test["isFraud"]

In [8]:
print("train:", X_train.shape, y_train.mean())
print("test:", X_test.shape, y_test.mean())

train: (2217905, 12) 0.0017832143396583713
test: (552504, 12) 0.007706731534975312


In [ ]:
# convert to parquet for faster read/write
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_parquet("../data/processed/X_train.parquet")
y_train.to_frame().to_parquet("../data/processed/y_train.parquet")
X_test.to_parquet("../data/processed/X_test.parquet")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet")

Action performed in this notebook

1. Filtered to TRANSFER and CASH_OUT only, since that’s where all the fraud is. Dropped nameOrig, nameDest and isFlaggedFraud. Result: 2,770,409 rows, 0.30% fraud, a much better ratio than the original 0.13%.
2. Engineered six features:
- errorBalanceOrig and errorBalanceDest: how far the balances are from what the transaction amount implies.
- origEmptied: the origin account was drained to 0.
- destWasEmpty: the destination account had a zero balance before.
- amountLog: log-scaled amount.
- hourOfDay: step % 24.
3. Sanity checked the features. Fraud averages origEmptied 0.98 vs 0.43 for legit, and destWasEmpty 0.65 vs 0.14. errorBalanceOrig is much lower for fraud (about 10.7k vs 287k), and errorBalanceDest is much higher (about 733k vs -31k). So the features separate the classes well.
4. One hot encoded type into type_TRANSFER.
5. Time-based 80/20 split on step, dropping step from the features. Train has 2,217,905 rows and test has 552,504.
6. Saved four Parquet files to data/processed/: X_train, y_train, X_test, y_test.